<h5>Comprueba el GPU

In [ ]:
!nvidia-smi

Thu Oct 22 23:36:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

<h5>Conexión con Google Drive (aloja la carpeta .zip con las imágenes)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


<h5> Copiamos y descomprimimos el dataset alojado en el disco local, ya que es más rapido

In [ ]:
!cp /content/drive/MyDrive/ProyectoSeminario/processed_v2.zip /content/
!unzip -q /content/processed_v2.zip -d /content/
!ls /content/processed

data.yaml  test  train	valid


<h5> Reemplazamos la ruta del data.yaml por la del colab

In [ ]:
from pathlib import Path

yaml_path = Path('/content/processed/data.yaml')
lines = yaml_path.read_text().splitlines()
lines = ['path: /content/processed' if l.startswith('path:') else l for l in lines]
yaml_path.write_text('\n'.join(lines) + '\n')

print(yaml_path.read_text())

path: /content/processed
train: train/images
val: valid/images
test: test/images

nc: 4
names: ['persona', 'poste_luz', 'senal_stop', 'senal_transito']



<h5> instalamos Ultralytics, que es una libreria que permite tabajar con modelos YOLO (el formato en el que desacargamos los 3 datasets).

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.7/112.6 GB disk)


<h5> entrenamos el baseline (primer modelo que nos sirve como punto de referencia).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/processed/data.yaml',
    imgsz=160,
    epochs=50,
    batch=64,
    patience=15,
    workers=2,
    project='/content/drive/MyDrive/ProyectoSeminario/runs',
    name='baseline_v1_160',
)

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/processed/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=160, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=baseline_v1_160-2, nbs=64, nms=None, opset=None, optimize=Fal

In [ ]:
best = YOLO('/content/drive/MyDrive/ProyectoSeminario/runs/baseline_v1_160/weights/best.pt')
metrics = best.val(data='/content/processed/data.yaml', imgsz=160, split='test')

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1413.5±856.7 MB/s, size: 39.0 KB)
val: Scanning /content/processed/test/labels... 1143 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1143/1143 2.1Kit/s 0.6s
val: /content/processed/test/images/pedestrian_000197.jpg: 1 duplicate labels removed
val: /content/processed/test/images/pedestrian_000313.jpg: 7 duplicate labels removed
val: New cache created: /content/processed/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 11.5it/s 6.2s
                   all       1143       1946      0.886      0.755      0.817      0.554
               persona        416        895       0.85      0.803      0.877      0.561
             poste_luz        449        749      0.802      0.466      0.556      0

<h5> más épocas - misma resolución. (a)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

model.train(
    data='/content/processed/data.yaml',
    imgsz=160,
    epochs=150,
    batch=64,
    patience=30,
    workers=2,
    project='/content/drive/MyDrive/ProyectoSeminario/runs',
    name='expA_160_e150',
)

Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/processed/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=160, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=expA_160_e150, nbs=64, nms=None, opset=None, optimize=False,

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7e713a13c830>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0

<h5> más épocas - más resolución. (b)

In [ ]:
from ultralytics import YOLO

#usar imgsz= 224
model = YOLO('yolov8n.pt')

model.train(
    data='/content/processed/data.yaml',
    imgsz=224,
    epochs=150,
    batch=64,
    patience=30,
    workers=2,
    project='/content/drive/MyDrive/ProyectoSeminario/runs',
    name='expB_224_e150',
)

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/processed/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=expB_224_e150-4, nbs=64, nms=None, opset=None, optimize=Fals

<h5> tres modelos compaativos

In [ ]:
from pathlib import Path
from ultralytics import YOLO

RUNS = Path('/content/drive/MyDrive/ProyectoSeminario/runs')
experimentos = [
    ('baseline (160, 50 ép.)', 'baseline_v1_160', 160),
    ('A (160, 150 ép.)', 'expA_160_e150', 160),
    ('B (224, 150 ép.)', 'expB_224_e150', 224),
]

for etiqueta, carpeta, imgsz in experimentos:
    pesos = RUNS / carpeta / 'weights' / 'best.pt'
    if not pesos.exists():
        print(f'\n{etiqueta}: no encontrado, se omite')
        continue

    m = YOLO(str(pesos)).val(
        data='/content/processed/data.yaml',
        imgsz=imgsz,
        split='test',
        plots=False,
        verbose=False,
    )

    print(f'\n=== {etiqueta} ===   mAP50 general: {m.box.map50:.3f}')
    print(f'{"clase":16s} {"P":>6s} {"R":>6s} {"mAP50":>7s}')
    for i, c in enumerate(m.ap_class_index):
        p, r, ap50, _ = m.box.class_result(i)
        print(f'{m.names[c]:16s} {p:6.3f} {r:6.3f} {ap50:7.3f}')


Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1300.1±539.4 MB/s, size: 28.3 KB)
val: Scanning /content/processed/test/labels.cache... 998 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 998/998 279.1Mit/s 0.0s
val: /content/processed/test/images/pedestrian_000197.jpg: 1 duplicate labels removed
val: /content/processed/test/images/pedestrian_000313.jpg: 7 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 14.7it/s 4.3s
                   all        998       1797      0.891       0.76      0.827      0.602
Speed: 0.1ms preprocess, 0.9ms inference, 0.0ms loss, 1.2ms postprocess per image

=== baseline (160, 50 ép.) ===   mAP50 general: 0.827
clase                 P      R   mAP50
persona           0.899  0.765   0.877
poste_luz     

Modelo elegido, B = IMGSZ = 224

Otro modelo

In [ ]:
from ultralytics import YOLO
best = YOLO('/content/drive/MyDrive/ProyectoSeminario/runs/expB_224_e150-4/weights/best.pt')
metrics = best.val(data='/content/processed/data.yaml', imgsz=224, split='test')

Ultralytics 8.4.184 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,133,436 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1481.5±796.5 MB/s, size: 39.0 KB)
val: Scanning /content/processed/test/labels.cache... 1143 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1143/1143 319.6Mit/s 0.0s
val: /content/processed/test/images/pedestrian_000197.jpg: 1 duplicate labels removed
val: /content/processed/test/images/pedestrian_000313.jpg: 7 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 72/72 10.7it/s 6.7s
                   all       1143       1946      0.887      0.827      0.868      0.648
               persona        416        895      0.858      0.828      0.891      0.597
             poste_luz        449        749      0.752      0.543      0.614      0.333
            senal_stop        158        162    